<a href="https://colab.research.google.com/github/laxmanchowdaryy/Space-Astronomy-BabyGPT/blob/main/01_Data_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests beautifulsoup4

In [2]:
import requests #downloading webpages
from bs4 import BeautifulSoup #To read HTML pages
import re #regular expressions for basic text cleaning.
import time
from urllib.parse import urljoin, urlparse

In [3]:
#Website sources
#In each source domain, starting urls are defined to start crawling

sources = {
    "NASA": {
        "domain": "science.nasa.gov",
        "start_urls": [
            "https://science.nasa.gov/universe/",
            "https://science.nasa.gov/exoplanets/",
            "https://science.nasa.gov/solar-system/"
        ]
    },

    "ISRO": {
        "domain": "isro.gov.in",
        "start_urls": [
            "https://www.isro.gov.in/"
        ]
    }
}

MAX_PAGES_PER_SOURCE = 200 #maximum no. of pages from each source to be collected

REQUEST_DELAY = 1 #waiting between each req

In [4]:
#keywords that could help searching for astronomy pages
astronomy_keywords = [
    "space",
    "astronomy",
    "astronom",
    "universe",
    "galaxy",
    "galaxies",
    "star",
    "stars",
    "black-hole",
    "black-hole",
    "exoplanet",
    "planet",
    "solar-system",
    "moon",
    "asteroid",
    "comet",
    "cosmos",
    "cosmology",
    "telescope",
    "observatory",
    "supernova",
    "nebula",
    "milky-way",
    "dark-matter",
    "dark-energy",
    "astrophysics",
    "satellite",
    "spacecraft",
    "space-science",
    "astronomical",
    "stellar"
]

In [15]:
#check if the url belong to the domain or not
def belongs_to_domain(url, domain):
    hostname = urlparse(url).netloc.lower() #extracting domain from url
    hostname = hostname.replace("www.", "") #ignoring www.
    domain = domain.lower().replace("www.", "") #removing www. from the website as well
    return hostname == domain or hostname.endswith("." + domain) #check if url matches the organization

In [16]:
#function checks if URL looks relevant to Space and Astronomy.
def is_astronomy_url(url, domain):
    if not belongs_to_domain(url, domain):  #if URL belongs to the selected organization.
        return False
    path = urlparse(url).path.lower() # Convert the URL path to lowercase.
    unwanted_extensions = [
        ".jpg", ".jpeg", ".png", ".gif",
        ".svg", ".webp", ".pdf", ".zip",
        ".mp3", ".mp4", ".avi", ".mov"
    ] # Ignore file types that are not HTML pages.
    for extension in unwanted_extensions:   # Check whether the URL points to an unwanted file.
        if path.endswith(extension):    # Reject the URL if it ends with one of these extensions.
            return False
    for keyword in astronomy_keywords: # Accept the page if an astronomy keyword appears in the URL.
        if keyword in path:  # Check the URL path for the keyword.
            return True
    return False

In [17]:
#downloads a webpage and extracts its text and links.
def scrape_page(url):
    try:
        # Download the webpage.
        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=20
        )
        # Check whether the webpage was successfully downloaded.
        if response.status_code != 200:
            return None, "", []
        # Convert the HTML into a BeautifulSoup object.
        soup = BeautifulSoup(response.text, "html.parser")
        # Get the webpage title.
        title = soup.title.get_text(" ", strip=True) if soup.title else ""
        # Remove elements that usually contain unwanted website information.
        for tag in soup([
            "script",
            "style",
            "nav",
            "header",
            "footer",
            "noscript",
            "form"
        ]):
            # Delete the unwanted HTML element.
            tag.decompose()
        # Try to find the main content.
        main_content = soup.find("main")
        # Use the main content when it is available.
        if main_content:
            text = main_content.get_text(separator=" ")
        # Otherwise extract text from the whole remaining page.
        else:
            text = soup.get_text(separator=" ")
        # Replace multiple spaces and line breaks with one space.
        text = re.sub(r"\s+", " ", text)
        # Remove unnecessary spaces from the beginning and end.
        text = text.strip()
        # Find all links on the webpage.
        links = soup.find_all("a", href=True)
        # Create a list for the links.
        page_links = []
        # Process every link.
        for link in links:
            href = link.get("href") #get link address
            full_url = urljoin(url, href)
            page_links.append(full_url)
        return text, title, page_links
    except Exception as error:
        print("Error:", url, error)
        return None, "", []

In [18]:
#function that crawls one organization at a time.
def crawl_source(source_name, source_info):
    domain = source_info["domain"] #specific org domain
    urls_to_visit = list(source_info["start_urls"])
    visited = set()
    source_documents = [] #for storing docs that are collected
    while urls_to_visit and len(source_documents) < MAX_PAGES_PER_SOURCE:
        current_url = urls_to_visit.pop(0)
        current_url = current_url.split("#")[0]  # Remove URL fragments.
        if current_url in visited:
            continue
        visited.add(current_url)
        if not belongs_to_domain(current_url, domain):
            continue
        print(
            source_name,
            "|",
            len(source_documents) + 1,
            "/",
            MAX_PAGES_PER_SOURCE,
            "|",
            current_url
        )

        text, title, page_links = scrape_page(current_url) #scrape the webpage
        if text and len(text) >= 500: #save if contains enough text
            source_documents.append({
                "source": source_name,
                "title": title,
                "url": current_url,
                "text": text
            })
        for link in page_links:
            link = link.split("#")[0] #remove url frags
            if (
                link not in visited
                and is_astronomy_url(link, domain)
            ):
                urls_to_visit.append(link)
        urls_to_visit = list(dict.fromkeys(urls_to_visit)) #remove duplicate urls
        time.sleep(REQUEST_DELAY) #wait time before continuing another req
    return source_documents

In [19]:
# Start collecting data from all selected Space and Astronomy sources.
all_documents = []
for source_name, source_info in sources.items(): #go thru each org
    print("\n========================================")
    print("Starting:", source_name)
    print("========================================\n")
    # Crawl the organization's astronomy pages.
    source_documents = crawl_source(
        source_name,
        source_info
    )
    # Add the collected documents to the main dataset.
    all_documents.extend(source_documents)
    print(
        "\nFinished",
        source_name,
        "- pages collected:",
        len(source_documents)
    )


Starting: NASA

NASA | 1 / 200 | https://science.nasa.gov/universe/
NASA | 2 / 200 | https://science.nasa.gov/exoplanets/
NASA | 3 / 200 | https://science.nasa.gov/solar-system/
NASA | 4 / 200 | https://science.nasa.gov/humans-in-space/
NASA | 5 / 200 | https://science.nasa.gov/moon/
NASA | 6 / 200 | https://science.nasa.gov/dwarf-planets/
NASA | 7 / 200 | https://science.nasa.gov/asteroids-comets-meteors/
NASA | 8 / 200 | https://science.nasa.gov/solar-system/kuiper-belt/
NASA | 9 / 200 | https://science.nasa.gov/solar-system/oort-cloud/
NASA | 10 / 200 | https://science.nasa.gov/universe/search-for-life/
NASA | 11 / 200 | https://science.nasa.gov/universe/stars/
NASA | 12 / 200 | https://science.nasa.gov/universe/galaxies/
NASA | 13 / 200 | https://science.nasa.gov/universe/black-holes/
NASA | 14 / 200 | https://science.nasa.gov/universe/the-big-bang/
NASA | 15 / 200 | https://science.nasa.gov/dark-matter/
NASA | 16 / 200 | https://science.nasa.gov/dark-energy/
NASA | 17 / 200 | htt

In [20]:
# Check the total amount of data collected.
print("========================================")
print("SCRAPING SUMMARY")
print("========================================")
# Print the total number of webpages.
print("Total webpages:", len(all_documents))
# Calculate the total number of characters.
total_characters = sum(
    len(document["text"])
    for document in all_documents
)
# Calculate the total number of words.
total_words = sum(
    len(document["text"].split())
    for document in all_documents
)
# Print the total characters and words.
print("Total characters:", total_characters)
print("Total words:", total_words)

SCRAPING SUMMARY
Total webpages: 243
Total characters: 1396067
Total words: 221071


In [21]:
# Remove duplicate documents based on their text.
# Store the original number of documents.
before_duplicates = len(all_documents)
# Create a set to keep track of text we have already seen.
seen_text = set()
# Create a list for unique documents.
unique_documents = []
for document in all_documents:
    text = document["text"]
    if text not in seen_text:
        seen_text.add(text)
        unique_documents.append(document)

# Replace the original dataset with the unique documents.
all_documents = unique_documents
# Calculate the number of duplicates removed.
duplicates_removed = before_duplicates - len(all_documents)
# Print the results.
print("Documents before duplicates:", before_duplicates)
print("Documents after duplicates:", len(all_documents))
print("Duplicates removed:", duplicates_removed)

Documents before duplicates: 243
Documents after duplicates: 240
Duplicates removed: 3


In [22]:
# Combine all collected and deduplicated text into one text corpus.
astronomy_text = "\n\n".join(
    document["text"]
    for document in all_documents
)
# Display the total size of the corpus.
print("Total characters:", len(astronomy_text))
print("Total words:", len(astronomy_text.split()))

Total characters: 1366133
Total words: 216196


In [23]:
# Display the beginning of the collected dataset. so we can check if its related to astronomy topic
print(astronomy_text[:10000])

Explore the Universe Welcome to the Universe Discover the universe: Learn about the history of the cosmos, what it's made of, and so much more. Explore: Exoplanets Worlds beyond our solar system. Stars Giant balls of hot gas that burn for millions to billions of years. Black Holes Concentrations of matter with gravity so powerful not even light can escape. Galaxies Collections of stars, planets, and vast clouds of gas and dust bound together by gravity. Featured Story NASA’s Dark Universe-Seeking Nancy Grace Roman Space Telescope Launches Now on a three-month, million-mile journey to its final orbit, NASA’s Nancy Grace Roman Space Telescope will soon reveal the… Read the Story A Universe of Fun and Learning Fun Exploring the Universe Astrobiology Learning Resources Online Activities with NASA’s Hubble Learn with NASA’s Webb Roman's Educational Activities Fun Exploring the Universe Explore our universe with these inspiring, informative, and fun activities and related resources. From col

In [24]:
# Save the collected text as a plain TXT file.
with open("Astronomy_raw.txt", "w", encoding="utf-8") as file:
    # Write the complete corpus to the file.
    file.write(astronomy_text)
print("Astronomy_raw.txt saved successfully.")

Astronomy_raw.txt saved successfully.


In [25]:
# Save a simple source list. so that we can able to add references and know where the exact data is collected from which site.
with open("astronomy_sources.txt", "w", encoding="utf-8") as file:
    # Write one source record for every collected webpage.
    for document in all_documents:
        file.write(
            document["source"]
            + " | "
            + document["title"]
            + " | "
            + document["url"]
            + "\n"
        )
print("astronomy_sources.txt saved successfully.")

astronomy_sources.txt saved successfully.
